# Visualisierung eines Noise2Void-Runs

In diesem Notebook gibst du **einmalig** den Pfad zu deinem gespeicherten Run-Ordner an.  
Dann werden daraus automatisch geladen:
- die `config.py` (aus `used_source`),  
- dein U-Net-Modell mit dem Checkpoint `best.pt`,  
- Trainings- und Validierungs-Datasets,  
- und es werden exemplarisch je drei Beispiele aus Training und Validation geplottet.

# Parameter setzen:

In [7]:
import os
import sys

# GPU wählen
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # or whichever GPU you want

# — Stelle hier deinen Run-Ordner ein (z.B. …/trained_models/First_Test) —
model = "ABC"  # as found in the trained_models directory
batch_size = 600
model_input = "../datasets/sf_brain_DMI_HC_pilot_normalized/data.npy" # this is the data that should be denoised by the model

Comparisons = [model_input] # in case you want to compare the model output to any other data sets, specify list of paths here

# Inference

In [11]:
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path

# Projekt-Root automatisch bestimmen
ROOT = Path().resolve()

# Falls Notebook im Projekt liegt:
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"

sys.path.insert(0, str(SRC))

print("Added to PYTHONPATH:", SRC)

from pathlib import Path

from denoising.config.load import load_yaml
from denoising.config.build import build_config
from denoising.inference.api import infer

config_path = f"../trained_models/{model}/train.yaml"

cfg = build_config(load_yaml(config_path))

denoised, meta = infer(
    cfg=cfg,
    ckpt_path=f"../trained_models/{model}/checkpoints/last.pt",
    input_path=model_input,   # exakter Pfad zur Datei
    batch_size=batch_size
    # KEIN output_path → nichts wird gespeichert
)

Added to PYTHONPATH: /workspace/Denoising/src


# Optional als matlab datei speichern

In [ ]:
from scipy.io import savemat

#savemat('sf_brain_DMI_HC_pilot_deep_only.mat', {'Data': out_data})

# savemat('Tumor_1_Part_1_tMPPCA_two_patients.mat', {'Data': out_data[...,:5]})
# savemat('Tumor_1_Part_2_tMPPCA_two_patients.mat', {'Data': out_data[...,5:]})

# savemat('Tumor_1_FullRank_Part1.mat', {'Data': out_data[...,:5]})
# savemat('Tumor_1_FullRank_Part2.mat', {'Data': out_data[...,5:]})

# savemat('Tumor_2_Part_1_LR8.mat', {'Data': baseline_data[...,:5]})
# savemat('Tumor_2_Part_2_LR8.mat', {'Data': baseline_data[...,5:]})

# savemat('P08_tMPPCA_5D.mat', {'Data': tgt_data[...,:5]})
# savemat('P08_tMPPCA_5D.mat', {'Data': tgt_data[...,5:]})

# savemat(f'{subject}_tMPPCA_5D.mat', {'Data': tgt_data})  
# savemat(f'{subject}_deep_tMPPCA_5D.mat', {'Data': out_data})

# Compare FID

In [ ]:
# ── Vergleich aller Z-Slices: Low-Rank | Deep Denoising | Input ─────────────

import numpy as np
import matplotlib.pyplot as plt

# 1) t- und T-Indizes einstellen
t, T = 15, 7

# 2) Anzahl der Z-Slices automatisch ermitteln
n_slices = out_data.shape[2]

# 3) Subplots erzeugen: n_slices Zeilen × 3 Spalten
fig, axes = plt.subplots(
    n_slices, 3,
    figsize=(9, n_slices * 2.5),
    constrained_layout=True
)

for i, z in enumerate(range(n_slices)):
    # 4) 2D-Slices extrahieren
    slice_lr   = np.abs(baseline_data[:, :, z, t, T])  # Low-Rank
    slice_deep = np.abs(out_data[:, :, z, t, T])       # Deep Denoised
    slice_in   = np.abs(tgt_data[:, :, z, t, T])       # Original Input

    # 5a) Low-Rank
    im0 = axes[i, 0].imshow(slice_lr,   cmap='viridis')
    axes[i, 0].set_title(f"low-rank {rank_post} | z={z}, t={t}")
    axes[i, 0].axis('on')
    axes[i, 0].grid(True, color='w', lw=0.5)  # weißes Grid
    plt.colorbar(im0, ax=axes[i, 0], fraction=0.046, pad=0.04)

    # 5b) Deep Denoising
    im1 = axes[i, 1].imshow(slice_deep, cmap='viridis')
    axes[i, 1].set_title(f"deep denoising' | z={z}, t={t}")
    axes[i, 1].axis('on')
    axes[i, 1].grid(True, color='w', lw=0.5)  # weißes Grid
    plt.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)

    # 5c) Input
    im2 = axes[i, 2].imshow(slice_in,   cmap='viridis')
    axes[i, 2].set_title(f"no denoising          | z={z}, t={t}")
    axes[i, 2].axis('on')
    axes[i, 2].grid(True, color='w', lw=0.5)  # weißes Grid

    plt.colorbar(im2, ax=axes[i, 2], fraction=0.046, pad=0.04)

#plt.savefig("denoising_p2n.png", dpi=300)
plt.show()

# Compare spectra

In [ ]:
x,y,T = 8, 10, 7

# 2a) Deep-Denoising Spektrum
spec_deep = np.fft.fft(out_data, axis=3)
spec_deep = np.fft.fftshift(spec_deep, axes=3)

# 2b) Noisy Input Spektrum
spec_noisy = np.fft.fft(tgt_data, axis=3)
spec_noisy = np.fft.fftshift(spec_noisy, axes=3)

# 2c) Low-Rank Baseline Spektrum
spec_lr = np.fft.fft(baseline_data, axis=3)
spec_lr = np.fft.fftshift(spec_lr, axes=3)

# ── 21 Spektren für z=0…20 in einem 5×5-Grid plotten ─────────────────────────

# ── 21 Spektren (Noisy vs. Low-Rank vs. Noise2Void) in 2 Spalten ────────────

# ── 21 Spektren in 2 Spalten mit eigener Legende pro Plot und größerer Figure ──

import numpy as np
import matplotlib.pyplot as plt

# Parameter
#x, y, T = 10, 10, 7
Z       = spec_noisy.shape[2]   # Anzahl der z-Slices (hier 21)
F       = spec_noisy.shape[3]   # Anzahl der Frequenz-Bins
freqs   = np.arange(F)
rank    = 8                     # Rang für Low-Rank

# Grid-Layout: 2 Spalten, genug Zeilen
n_cols  = 2
n_rows  = int(np.ceil(Z / n_cols))

# Figure größer machen: Breite × Höhe in Zoll
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(12, n_rows * 3),
    sharex=True, sharey=True,
    constrained_layout=True
)

for z in range(Z):
    i, j = divmod(z, n_cols)
    ax = axes[i, j]

    # Magnituden extrahieren
    mag_noisy = np.abs(spec_noisy[x, y, z, :, T])
    mag_lr    = np.abs(spec_lr   [x, y, z, :, T])
    mag_deep  = np.abs(spec_deep [x, y, z, :, T])

    # Plots
    ax.plot(freqs, mag_noisy, '-',  label='no denosing', linewidth=1)
    ax.plot(freqs, mag_lr,    '-', label=f'Low-Rank (r={rank_post})', linewidth=1)
    ax.plot(freqs, mag_deep,  '-',  label=f'deep denoising', linewidth=1)

    ax.set_title(f"z={z}")
    ax.grid(True, linestyle=':', alpha=0.3)

    # Legende für jeden Subplot
    ax.legend(fontsize='small', loc='upper right')

    # Achsenbeschriftungen nur außen
    if i == n_rows - 1:
        ax.set_xlabel("Frequency bin")
    if j == 0:
        ax.set_ylabel("Magnitude")

# Leere Subplots ausblenden
for idx in range(Z, n_rows * n_cols):
    i, j = divmod(idx, n_cols)
    axes[i, j].axis('off')

#plt.savefig("spectra.png", dpi=300)
plt.show()



In [ ]:
x,y,T = 10, 10, 0

# 2a) Deep-Denoising Spektrum
spec_deep = np.fft.fft(out_data, axis=3)
spec_deep = np.fft.fftshift(spec_deep, axes=3)

# 2b) Noisy Input Spektrum
spec_noisy = np.fft.fft(tgt_data, axis=3)
spec_noisy = np.fft.fftshift(spec_noisy, axes=3)

# 2c) Low-Rank Baseline Spektrum
spec_lr = np.fft.fft(baseline_data, axis=3)
spec_lr = np.fft.fftshift(spec_lr, axes=3)

# ── 21 Spektren für z=0…20 in einem 5×5-Grid plotten ─────────────────────────

# ── 21 Spektren (Noisy vs. Low-Rank vs. Noise2Void) in 2 Spalten ────────────

# ── 21 Spektren in 2 Spalten mit eigener Legende pro Plot und größerer Figure ──

import numpy as np
import matplotlib.pyplot as plt

# Parameter
x, y, T = 10, 10, 7
Z       = spec_noisy.shape[2]   # Anzahl der z-Slices (hier 21)
F       = spec_noisy.shape[3]   # Anzahl der Frequenz-Bins
freqs   = np.arange(F)
rank    = 8                     # Rang für Low-Rank

# Grid-Layout: 2 Spalten, genug Zeilen
n_cols  = 2
n_rows  = int(np.ceil(Z / n_cols))

# Figure größer machen: Breite × Höhe in Zoll
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(12, n_rows * 3),
    sharex=True, sharey=True,
    constrained_layout=True
)

for z in range(Z):
    i, j = divmod(z, n_cols)
    ax = axes[i, j]

    # Magnituden extrahieren
    mag_noisy = np.abs(spec_noisy[x, y, z, :, T])
    mag_lr    = np.abs(spec_lr   [x, y, z, :, T])
    mag_deep  = np.abs(spec_deep [x, y, z, :, T])

    # Plots
    ax.plot(freqs, mag_lr,    '-', label=f'Low-Rank (r={rank_post})', linewidth=1)
    ax.plot(freqs, mag_deep,  '-',  label=f'deep denoising', linewidth=1)

    ax.set_title(f"z={z}")
    ax.grid(True, linestyle=':', alpha=0.3)

    # Legende für jeden Subplot
    ax.legend(fontsize='small', loc='upper right')

    # Achsenbeschriftungen nur außen
    if i == n_rows - 1:
        ax.set_xlabel("Frequency bin")
    if j == 0:
        ax.set_ylabel("Magnitude")

# Leere Subplots ausblenden
for idx in range(Z, n_rows * n_cols):
    i, j = divmod(idx, n_cols)
    axes[i, j].axis('off')

#plt.savefig("spectra.png", dpi=300)
plt.show()

# Compare spectral peaks

In [ ]:
# ── Vergleich aller Z-Slices: Low-Rank | Deep Denoising | Input ─────────────

import numpy as np
import matplotlib.pyplot as plt

# 1) t- und T-Indizes einstellen
t, T = 78, 7  # 85~Glx

# 2) Anzahl der Z-Slices automatisch ermitteln
n_slices = out_data.shape[2]

# 3) Subplots erzeugen: n_slices Zeilen × 3 Spalten
fig, axes = plt.subplots(
    n_slices, 3,
    figsize=(9, n_slices * 2.5),
    constrained_layout=True
)

for i, z in enumerate(range(n_slices)):
    # 4) 2D-Slices extrahieren
    slice_lr   = np.abs(baseline_data_ft[:, :, z, t, T])  # Low-Rank
    slice_deep = np.abs(out_data_ft[:, :, z, t, T])       # Deep Denoised
    slice_in   = np.abs(tgt_data_ft[:, :, z, t, T])       # Original Input

    # 5a) Low-Rank
    im0 = axes[i, 0].imshow(slice_lr,   cmap='viridis')
    axes[i, 0].set_title(f"low-rank {rank_post} | z={z}, t={t}")
    #axes[i, 0].axis('off')
    axes[i, 0].axis('on')
    axes[i, 0].grid(True, color='w', lw=0.5)  # weißes Grid
    plt.colorbar(im0, ax=axes[i, 0], fraction=0.046, pad=0.04)

    # 5b) Deep Denoising
    im1 = axes[i, 1].imshow(slice_deep, cmap='viridis')
    axes[i, 1].set_title(f"deep denoising | z={z}, t={t}")
    #axes[i, 1].axis('off')
    axes[i, 1].axis('on')
    axes[i, 1].grid(True, color='w', lw=0.5)  # weißes Grid
    plt.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)

    # 5c) Input
    im2 = axes[i, 2].imshow(slice_in,   cmap='viridis')
    axes[i, 2].set_title(f"no denoising          | z={z}, t={t}")
    #axes[i, 2].axis('off')
    axes[i, 2].axis('on')
    axes[i, 2].grid(True, color='w', lw=0.5)  # weißes Grid
    plt.colorbar(im2, ax=axes[i, 2], fraction=0.046, pad=0.04)

#plt.savefig("denoising_p2n.png", dpi=300)
plt.show()

# Compare average spectra
Here I compare the average spectrum over time (which is a high SNR estimate) for gey matter which matter and all matter

In [ ]:
avg_out = np.mean(out_data_ft, axis=(0, 1, 2))

avg_lr = np.mean(baseline_data_ft, axis=(0, 1, 2))

avg_tgt = np.mean(tgt_data_ft, axis=(0, 1, 2))

In [ ]:
import matplotlib.pyplot as plt

# Subplot-Grid: 4 Zeilen × 2 Spalten für T = 0 bis 7
num_rows, num_cols = 5, 2
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 16))

for idx, T in enumerate(range(8)):
    i = idx // num_cols
    j = idx % num_cols
    ax = axes[i, j]
    
    # Spektren extrahieren
    Line   = np.abs(avg_out)[:, T]
    Line_2 = np.abs(avg_tgt)[:, T]
    Line_3 = np.abs(avg_lr)[:, T]
    
    ax.plot(Line,   label="Deep Denoising")
    ax.plot(Line_2, linestyle="--", label="Original (Noisy)")
    ax.plot(Line_3, linestyle=":",  label="Low-Rank")
    
    ax.set_xlabel("FID-Zeitpunkt (t)")
    ax.set_ylabel("Betragsspektrum")
    ax.set_title(f"T = {T}")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()
